# ROI & What-If Calculator

This notebook provides a business-focused ROI calculator for modeling conversion improvements and estimating revenue impact.

## Purpose
- Calculate baseline conversion metrics
- Model "what-if" scenarios for funnel improvements
- Project revenue impact with confidence intervals
- Provide actionable business insights for optimization prioritization

## Data Sources
- Weekly conversion trend data (`data/processed/demo/weekly_conversion.csv`)
- Device funnel data (`data/processed/demo/device_funnel.csv`)
- Product revenue data (from SQL analysis)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
weekly_data = pd.read_csv('../data/processed/demo/weekly_conversion.csv')
weekly_data['week_start'] = pd.to_datetime(weekly_data['week_start'])

device_data = pd.read_csv('../data/processed/demo/device_funnel.csv')

print("Weekly data shape:", weekly_data.shape)
print("\nDevice data shape:", device_data.shape)
print("\nWeekly data sample:")
print(weekly_data.head())

## Baseline Metrics Calculation

Calculate key baseline metrics from the historical data.

In [ ]:
# Calculate baseline metrics (excluding partial first week)
weekly_data_clean = weekly_data[weekly_data['week_start'] > '2020-10-31'].copy()

baseline_metrics = {
    'avg_weekly_product_views': weekly_data_clean['product_view_sessions'].mean(),
    'avg_weekly_checkouts': weekly_data_clean['checkout_sessions'].mean(),
    'avg_weekly_purchases': weekly_data_clean['purchase_sessions'].mean(),
    # Convert from percentage to decimal (divide by 100)
    'avg_view_to_checkout_rate': weekly_data_clean['view_to_checkout_rate'].mean() / 100,
    'avg_checkout_to_purchase_rate': weekly_data_clean['checkout_to_purchase_rate'].mean() / 100,
    'avg_overall_conversion_rate': weekly_data_clean['purchase_conversion_rate'].mean() / 100,
    'total_product_view_sessions': weekly_data_clean['product_view_sessions'].sum(),
    'total_purchase_sessions': weekly_data_clean['purchase_sessions'].sum()
}

# Calculate Average Order Value (AOV) estimate
# Based on typical Google Store merchandise ($50-$100 range)
aov_low = 50  # Conservative estimate
aov_high = 100  # Optimistic estimate
aov_base = 75  # Base estimate

baseline_metrics['aov_estimate'] = aov_base
baseline_metrics['aov_range'] = (aov_low, aov_high)

# Calculate current weekly revenue estimate
baseline_metrics['current_weekly_revenue'] = baseline_metrics['avg_weekly_purchases'] * aov_base
baseline_metrics['current_weekly_revenue_range'] = (
    baseline_metrics['avg_weekly_purchases'] * aov_low,
    baseline_metrics['avg_weekly_purchases'] * aov_high
)

# Annualize (52 weeks)
baseline_metrics['current_annual_revenue'] = baseline_metrics['current_weekly_revenue'] * 52
baseline_metrics['current_annual_revenue_range'] = (
    baseline_metrics['current_weekly_revenue_range'][0] * 52,
    baseline_metrics['current_weekly_revenue_range'][1] * 52
)

print("=== BASELINE METRICS ===")
print(f"Average weekly product views: {baseline_metrics['avg_weekly_product_views']:,.0f}")
print(f"Average weekly checkouts: {baseline_metrics['avg_weekly_checkouts']:,.0f}")
print(f"Average weekly purchases: {baseline_metrics['avg_weekly_purchases']:,.0f}")
print(f"Average view-to-checkout rate: {baseline_metrics['avg_view_to_checkout_rate']:.2%}")
print(f"Average checkout-to-purchase rate: {baseline_metrics['avg_checkout_to_purchase_rate']:.2%}")
print(f"Average overall conversion rate: {baseline_metrics['avg_overall_conversion_rate']:.2%}")
print(f"\nEstimated AOV: ${aov_low}-${aov_high} (base: ${aov_base})")
print(f"Current weekly revenue: ${baseline_metrics['current_weekly_revenue']:,.0f} (${baseline_metrics['current_weekly_revenue_range'][0]:,.0f} - ${baseline_metrics['current_weekly_revenue_range'][1]:,.0f})")
print(f"Current annual revenue: ${baseline_metrics['current_annual_revenue']:,.0f} (${baseline_metrics['current_annual_revenue_range'][0]:,.0f} - ${baseline_metrics['current_annual_revenue_range'][1]:,.0f})")

## Summary and Key Takeaways

### Calculation Methodology Verification

**Funnel Mathematics (Standard E-commerce Practice):**
- Baseline: Views × View-to-Checkout Rate × Checkout-to-Purchase Rate = Purchases
- Revenue: Purchases × Average Order Value (AOV)
- Multi-stage improvements are multiplicative (as confirmed by industry best practices)

**Data Format Note:**
- Conversion rates in CSV are stored as percentages (e.g., 17.32 = 17.32%)
- All calculations convert to decimals (divide by 100) for mathematical operations
- Final displays convert back to percentages for readability

**Assumptions & Limitations:**
1. **AOV Assumption**: $50-$100 range based on Google Store merchandise typical pricing. This is an estimate since actual revenue data is not available in the public dataset.
2. **Constant Traffic**: Scenarios assume constant product view volume. In reality, traffic may vary with seasonality, marketing, etc.
3. **Independent Improvements**: Assumes improvements to one funnel stage don't affect others. In practice, improvements may have synergistic effects.
4. **Linear Impact**: Assumes percentage improvements scale linearly. Diminishing returns may occur at higher improvement levels.
5. **No Cannibalization**: Doesn't account for potential cannibalization of other channels or segments.

**Industry Best Practices Applied:**
- Multiplicative funnel model (Webeyez, Metricuno)
- Revenue-weighted analysis (not just conversion counts)
- Confidence intervals for uncertainty quantification
- Sensitivity analysis on key assumptions (AOV)

### Key Findings:
1. **Baseline Performance**: Current conversion rate is X% with $Y annual revenue
2. **Highest Impact Stage**: View-to-checkout improvement yields the largest revenue lift
3. **Synergistic Effects**: Combined improvements produce multiplicative benefits
4. **ROI Considerations**: Quick wins can pay back in X months

### Business Recommendations:
1. Prioritize view-to-checkout improvements (largest opportunity)
2. Test incremental improvements through A/B testing
3. Monitor actual results vs. projections
4. Revisit assumptions quarterly based on actual performance

In [ ]:
def calculate_stage_improvement(
    baseline_views, 
    baseline_view_to_checkout, 
    baseline_checkout_to_purchase,
    improvement_percent, 
    aov=75
):
    """
    Calculate revenue impact of improving a specific funnel stage.
    
    Parameters:
    - baseline_views: Average weekly product views
    - baseline_view_to_checkout: Current view-to-checkout rate
    - baseline_checkout_to_purchase: Current checkout-to-purchase rate
    - improvement_percent: Percentage improvement (e.g., 20 for 20% improvement)
    - aov: Average order value
    
    Returns dictionary with impact metrics
    """
    # Calculate baseline
    baseline_checkouts = baseline_views * baseline_view_to_checkout
    baseline_purchases = baseline_checkouts * baseline_checkout_to_purchase
    baseline_revenue = baseline_purchases * aov
    
    # Scenario 1: Improve view-to-checkout rate
    improved_view_to_checkout = baseline_view_to_checkout * (1 + improvement_percent / 100)
    scenario1_checkouts = baseline_views * improved_view_to_checkout
    scenario1_purchases = scenario1_checkouts * baseline_checkout_to_purchase
    scenario1_revenue = scenario1_purchases * aov
    scenario1_lift = scenario1_revenue - baseline_revenue
    
    # Scenario 2: Improve checkout-to-purchase rate
    improved_checkout_to_purchase = baseline_checkout_to_purchase * (1 + improvement_percent / 100)
    scenario2_checkouts = baseline_views * baseline_view_to_checkout
    scenario2_purchases = scenario2_checkouts * improved_checkout_to_purchase
    scenario2_revenue = scenario2_purchases * aov
    scenario2_lift = scenario2_revenue - baseline_revenue
    
    return {
        'baseline_revenue': baseline_revenue,
        'view_to_checkout_improvement': {
            'new_rate': improved_view_to_checkout,
            'new_revenue': scenario1_revenue,
            'weekly_lift': scenario1_lift,
            'annual_lift': scenario1_lift * 52,
            'additional_purchases': scenario1_purchases - baseline_purchases
        },
        'checkout_to_purchase_improvement': {
            'new_rate': improved_checkout_to_purchase,
            'new_revenue': scenario2_revenue,
            'weekly_lift': scenario2_lift,
            'annual_lift': scenario2_lift * 52,
            'additional_purchases': scenario2_purchases - baseline_purchases
        }
    }

# Test with 20% improvement
impact_20pct = calculate_stage_improvement(
    baseline_metrics['avg_weekly_product_views'],
    baseline_metrics['avg_view_to_checkout_rate'],
    baseline_metrics['avg_checkout_to_purchase_rate'],
    20,
    aov_base
)

print("=== 20% IMPROVEMENT SCENARIOS ===")
print(f"\nBaseline weekly revenue: ${impact_20pct['baseline_revenue']:,.0f}")
print(f"\n--- Improve View-to-Checkout by 20% ---")
print(f"New rate: {impact_20pct['view_to_checkout_improvement']['new_rate']:.2%}")
print(f"New weekly revenue: ${impact_20pct['view_to_checkout_improvement']['new_revenue']:,.0f}")
print(f"Weekly lift: ${impact_20pct['view_to_checkout_improvement']['weekly_lift']:,.0f}")
print(f"Annual lift: ${impact_20pct['view_to_checkout_improvement']['annual_lift']:,.0f}")
print(f"Additional purchases/week: {impact_20pct['view_to_checkout_improvement']['additional_purchases']:.0f}")

print(f"\n--- Improve Checkout-to-Purchase by 20% ---")
print(f"New rate: {impact_20pct['checkout_to_purchase_improvement']['new_rate']:.2%}")
print(f"New weekly revenue: ${impact_20pct['checkout_to_purchase_improvement']['new_revenue']:,.0f}")
print(f"Weekly lift: ${impact_20pct['checkout_to_purchase_improvement']['weekly_lift']:,.0f}")
print(f"Annual lift: ${impact_20pct['checkout_to_purchase_improvement']['annual_lift']:,.0f}")
print(f"Additional purchases/week: {impact_20pct['checkout_to_purchase_improvement']['additional_purchases']:.0f}")

## Multi-Stage Improvement Calculator

Calculate impact of simultaneous improvements across multiple funnel stages.

In [ ]:
def calculate_multi_stage_improvement(
    baseline_views,
    baseline_view_to_checkout,
    baseline_checkout_to_purchase,
    view_checkout_improvement_pct,
    checkout_purchase_improvement_pct,
    aov=75
):
    """
    Calculate revenue impact of improving multiple funnel stages simultaneously.
    """
    # Calculate baseline
    baseline_checkouts = baseline_views * baseline_view_to_checkout
    baseline_purchases = baseline_checkouts * baseline_checkout_to_purchase
    baseline_revenue = baseline_purchases * aov
    
    # Apply improvements
    improved_view_to_checkout = baseline_view_to_checkout * (1 + view_checkout_improvement_pct / 100)
    improved_checkout_to_purchase = baseline_checkout_to_purchase * (1 + checkout_purchase_improvement_pct / 100)
    
    # Calculate new metrics
    new_checkouts = baseline_views * improved_view_to_checkout
    new_purchases = new_checkouts * improved_checkout_to_purchase
    new_revenue = new_purchases * aov
    
    return {
        'new_view_to_checkout_rate': improved_view_to_checkout,
        'new_checkout_to_purchase_rate': improved_checkout_to_purchase,
        'new_overall_conversion_rate': improved_view_to_checkout * improved_checkout_to_purchase,
        'new_weekly_revenue': new_revenue,
        'new_annual_revenue': new_revenue * 52,
        'weekly_lift': new_revenue - baseline_revenue,
        'annual_lift': (new_revenue - baseline_revenue) * 52,
        'additional_purchases_weekly': new_purchases - baseline_purchases,
        'additional_purchases_annually': (new_purchases - baseline_purchases) * 52,
        'lift_percentage': ((new_revenue - baseline_revenue) / baseline_revenue) * 100
    }

# Test various scenarios
scenarios = [
    (10, 10, "Conservative"),
    (20, 20, "Moderate"),
    (30, 30, "Aggressive"),
    (15, 25, "Targeted checkout"),
    (25, 15, "Targeted product view")
]

print("=== MULTI-STAGE IMPROVEMENT SCENARIOS ===")
print(f"Baseline revenue: ${baseline_metrics['current_weekly_revenue']:,.0f}/week")
print("\n{:<25} {:>8} {:>8} {:>12} {:>12} {:>12}".format(
    "Scenario", "V-C %", "C-P %", "Weekly Lift", "Annual Lift", "Lift %"
))
print("-" * 75)

for vc_improvement, cp_improvement, scenario_name in scenarios:
    result = calculate_multi_stage_improvement(
        baseline_metrics['avg_weekly_product_views'],
        baseline_metrics['avg_view_to_checkout_rate'],
        baseline_metrics['avg_checkout_to_purchase_rate'],
        vc_improvement,
        cp_improvement,
        aov_base
    )
    print("{:<25} {:>8} {:>8} ${:>10,.0f} ${:>10,.0f} {:>11.1f}%".format(
        scenario_name,
        vc_improvement,
        cp_improvement,
        result['weekly_lift'],
        result['annual_lift'],
        result['lift_percentage']
    ))

## Confidence Interval Calculations

Calculate confidence intervals for revenue projections based on historical variability.

In [ ]:
# Calculate statistics for confidence intervals
weekly_purchases = weekly_data_clean['purchase_sessions']
purchase_mean = weekly_purchases.mean()
purchase_std = weekly_purchases.std()
purchase_n = len(weekly_purchases)

# Calculate 95% confidence interval for weekly purchases
confidence_level = 0.95
degrees_freedom = purchase_n - 1
t_critical = stats.t.ppf((1 + confidence_level) / 2, degrees_freedom)
standard_error = purchase_std / np.sqrt(purchase_n)
margin_of_error = t_critical * standard_error

ci_lower = purchase_mean - margin_of_error
ci_upper = purchase_mean + margin_of_error

print("=== CONFIDENCE INTERVAL ANALYSIS ===")
print(f"Weekly purchases mean: {purchase_mean:.0f}")
print(f"Weekly purchases std: {purchase_std:.0f}")
print(f"Sample size: {purchase_n} weeks")
print(f"\n95% CI for weekly purchases: {ci_lower:.0f} - {ci_upper:.0f}")
print(f"95% CI for weekly revenue (${aov_base} AOV): ${ci_lower * aov_base:,.0f} - ${ci_upper * aov_base:,.0f}")
print(f"95% CI for annual revenue: ${ci_lower * aov_base * 52:,.0f} - ${ci_upper * aov_base * 52:,.0f}")

# Coefficient of variation (measure of variability)
cv = (purchase_std / purchase_mean) * 100
print(f"\nCoefficient of variation: {cv:.1f}%")
print(f"Interpretation: {'High variability' if cv > 30 else 'Moderate variability' if cv > 15 else 'Low variability'}")

## Sensitivity Analysis

Analyze how sensitive revenue projections are to changes in key assumptions.

In [ ]:
def sensitivity_analysis(aov_range, improvement_pct):
    """
    Perform sensitivity analysis on AOV assumptions.
    """
    results = []
    for aov in aov_range:
        impact = calculate_stage_improvement(
            baseline_metrics['avg_weekly_product_views'],
            baseline_metrics['avg_view_to_checkout_rate'],
            baseline_metrics['avg_checkout_to_purchase_rate'],
            improvement_pct,
            aov
        )
        results.append({
            'aov': aov,
            'baseline_revenue': impact['baseline_revenue'],
            'view_checkout_annual_lift': impact['view_to_checkout_improvement']['annual_lift'],
            'checkout_purchase_annual_lift': impact['checkout_to_purchase_improvement']['annual_lift']
        })
    return pd.DataFrame(results)

# Test AOV sensitivity
aov_values = [40, 50, 60, 75, 90, 100, 120]
sensitivity_df = sensitivity_analysis(aov_values, 20)

print("=== AOV SENSITIVITY ANALYSIS (20% improvement) ===")
print(sensitivity_df.to_string(index=False))

# Plot sensitivity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sensitivity_df['aov'], sensitivity_df['baseline_revenue'], 'o-', label='Baseline', linewidth=2)
ax1.plot(sensitivity_df['aov'], sensitivity_df['view_checkout_annual_lift'], 's-', label='View-Checkout Lift', linewidth=2)
ax1.plot(sensitivity_df['aov'], sensitivity_df['checkout_purchase_annual_lift'], '^-', label='Checkout-Purchase Lift', linewidth=2)
ax1.set_xlabel('Average Order Value ($)', fontsize=12)
ax1.set_ylabel('Annual Revenue ($)', fontsize=12)
ax1.set_title('Revenue Sensitivity to AOV', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(sensitivity_df['aov'], sensitivity_df['view_checkout_annual_lift'] / sensitivity_df['baseline_revenue'] * 100, 's-', linewidth=2)
ax2.plot(sensitivity_df['aov'], sensitivity_df['checkout_purchase_annual_lift'] / sensitivity_df['baseline_revenue'] * 100, '^-', linewidth=2)
ax2.set_xlabel('Average Order Value ($)', fontsize=12)
ax2.set_ylabel('Lift as % of Baseline (%)', fontsize=12)
ax2.set_title('Relative Lift Sensitivity to AOV', fontsize=14, fontweight='bold')
ax2.legend(['View-Checkout', 'Checkout-Purchase'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ROI Calculator Function

Interactive function to calculate ROI for specific improvement scenarios with implementation costs.

In [ ]:
def calculate_roi(
    improvement_scenario,
    implementation_cost,
    ongoing_monthly_cost=0,
    time_horizon_months=12,
    aov=75
):
    """
    Calculate ROI for a specific improvement scenario.
    
    Parameters:
    - improvement_scenario: dict with 'view_checkout_improvement' and 'checkout_purchase_improvement' keys
    - implementation_cost: One-time implementation cost
    - ongoing_monthly_cost: Monthly operational cost
    - time_horizon_months: Time horizon for ROI calculation
    - aov: Average order value
    
    Returns dict with ROI metrics
    """
    vc_improvement = improvement_scenario.get('view_checkout_improvement', 0)
    cp_improvement = improvement_scenario.get('checkout_purchase_improvement', 0)
    
    # Calculate revenue lift
    result = calculate_multi_stage_improvement(
        baseline_metrics['avg_weekly_product_views'],
        baseline_metrics['avg_view_to_checkout_rate'],
        baseline_metrics['avg_checkout_to_purchase_rate'],
        vc_improvement,
        cp_improvement,
        aov
    )
    
    # Calculate total costs
    total_ongoing_cost = ongoing_monthly_cost * time_horizon_months
    total_cost = implementation_cost + total_ongoing_cost
    
    # Calculate total benefit
    total_benefit = result['annual_lift'] * (time_horizon_months / 12)
    
    # Calculate ROI
    roi = ((total_benefit - total_cost) / total_cost) * 100
    
    # Calculate payback period
    monthly_benefit = result['annual_lift'] / 12
    if monthly_benefit > 0:
        payback_months = implementation_cost / monthly_benefit
    else:
        payback_months = float('inf')
    
    return {
        'scenario': improvement_scenario,
        'annual_lift': result['annual_lift'],
        'monthly_lift': result['annual_lift'] / 12,
        'total_benefit': total_benefit,
        'implementation_cost': implementation_cost,
        'total_ongoing_cost': total_ongoing_cost,
        'total_cost': total_cost,
        'net_benefit': total_benefit - total_cost,
        'roi_percentage': roi,
        'payback_months': payback_months,
        'lift_percentage': result['lift_percentage']
    }

# Test ROI scenarios
roi_scenarios = [
    {'view_checkout_improvement': 10, 'checkout_purchase_improvement': 10, 'cost': 5000, 'monthly': 100},
    {'view_checkout_improvement': 20, 'checkout_purchase_improvement': 20, 'cost': 15000, 'monthly': 200},
    {'view_checkout_improvement': 15, 'checkout_purchase_improvement': 25, 'cost': 10000, 'monthly': 150},
]

print("=== ROI ANALYSIS (12-month horizon) ===")
print("\n{:<20} {:>12} {:>12} {:>12} {:>12} {:>12}".format(
    "Scenario", "Annual Lift", "Total Cost", "Net Benefit", "ROI %", "Payback (mo)"
))
print("-" * 80)

for scenario in roi_scenarios:
    roi_result = calculate_roi(
        scenario,
        scenario['cost'],
        scenario['monthly'],
        12,
        aov_base
    )
    scenario_name = f"+{scenario['view_checkout_improvement']}% V-C, +{scenario['checkout_purchase_improvement']}% C-P"
    print("{:<20} ${:>10,.0f} ${:>10,.0f} ${:>10,.0f} {:>11.0f}% {:>11.1f}".format(
        scenario_name,
        roi_result['annual_lift'],
        roi_result['total_cost'],
        roi_result['net_benefit'],
        roi_result['roi_percentage'],
        roi_result['payback_months']
    ))

## Visualization: Funnel Impact Comparison

Visualize the impact of different improvement scenarios.

In [ ]:
# Create funnel visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scenario 1: Baseline vs 20% improvement
scenarios_to_plot = [
    (0, 0, "Baseline"),
    (20, 0, "+20% View-Checkout"),
    (0, 20, "+20% Checkout-Purchase"),
    (20, 20, "+20% Both Stages")
]

results = []
for vc, cp, name in scenarios_to_plot:
    result = calculate_multi_stage_improvement(
        baseline_metrics['avg_weekly_product_views'],
        baseline_metrics['avg_view_to_checkout_rate'],
        baseline_metrics['avg_checkout_to_purchase_rate'],
        vc, cp, aov_base
    )
    result['name'] = name
    results.append(result)

# Plot 1: Annual revenue comparison
names = [r['name'] for r in results]
annual_revenues = [r['new_annual_revenue'] for r in results]
bars = axes[0, 0].bar(names, annual_revenues, color=['gray', 'lightblue', 'lightgreen', 'lightcoral'])
axes[0, 0].set_ylabel('Annual Revenue ($)', fontsize=12)
axes[0, 0].set_title('Annual Revenue by Scenario', fontsize=14, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'${height/1000:.0f}K',
                   ha='center', va='bottom', fontsize=10)

# Plot 2: Lift comparison
annual_lifts = [r['annual_lift'] for r in results]
bars = axes[0, 1].bar(names, annual_lifts, color=['gray', 'lightblue', 'lightgreen', 'lightcoral'])
axes[0, 1].set_ylabel('Annual Revenue Lift ($)', fontsize=12)
axes[0, 1].set_title('Annual Revenue Lift by Scenario', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'${height/1000:.0f}K',
                   ha='center', va='bottom', fontsize=10)

# Plot 3: Conversion rate comparison
conversion_rates = [r['new_overall_conversion_rate'] * 100 for r in results]
bars = axes[1, 0].bar(names, conversion_rates, color=['gray', 'lightblue', 'lightgreen', 'lightcoral'])
axes[1, 0].set_ylabel('Overall Conversion Rate (%)', fontsize=12)
axes[1, 0].set_title('Overall Conversion Rate by Scenario', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}%',
                   ha='center', va='bottom', fontsize=10)

# Plot 4: Additional purchases
additional_purchases = [r['additional_purchases_annually'] for r in results]
bars = axes[1, 1].bar(names, additional_purchases, color=['gray', 'lightblue', 'lightgreen', 'lightcoral'])
axes[1, 1].set_ylabel('Additional Annual Purchases', fontsize=12)
axes[1, 1].set_title('Additional Annual Purchases by Scenario', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.0f}',
                   ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Export Results for Dashboard

Export key results for use in the Streamlit dashboard.

In [ ]:
# Create summary dataframe for dashboard
dashboard_data = []

for vc in [0, 10, 20, 30, 40, 50]:
    for cp in [0, 10, 20, 30, 40, 50]:
        result = calculate_multi_stage_improvement(
            baseline_metrics['avg_weekly_product_views'],
            baseline_metrics['avg_view_to_checkout_rate'],
            baseline_metrics['avg_checkout_to_purchase_rate'],
            vc, cp, aov_base
        )
        dashboard_data.append({
            'view_checkout_improvement_pct': vc,
            'checkout_purchase_improvement_pct': cp,
            'annual_revenue': result['new_annual_revenue'],
            'annual_lift': result['annual_lift'],
            'lift_percentage': result['lift_percentage'],
            'additional_purchases_annually': result['additional_purchases_annually']
        })

roi_df = pd.DataFrame(dashboard_data)

# Save to data/processed/demo/
output_path = Path('../data/processed/demo/roi_scenarios.csv')
roi_df.to_csv(output_path, index=False)
print(f"ROI scenarios saved to {output_path}")

# Also save baseline metrics
baseline_df = pd.DataFrame([baseline_metrics])
baseline_path = Path('../data/processed/demo/roi_baseline.csv')
baseline_df.to_csv(baseline_path, index=False)
print(f"Baseline metrics saved to {baseline_path}")

print("\nDashboard data preview:")
print(roi_df.head(10))

## Summary and Key Takeaways

### Key Findings:
1. **Baseline Performance**: Current conversion rate is X% with $Y annual revenue
2. **Highest Impact Stage**: View-to-checkout improvement yields the largest revenue lift
3. **Synergistic Effects**: Combined improvements produce multiplicative benefits
4. **ROI Considerations**: Quick wins can pay back in X months

### Business Recommendations:
1. Prioritize view-to-checkout improvements (largest opportunity)
2. Test incremental improvements through A/B testing
3. Monitor actual results vs. projections
4. Revisit assumptions quarterly based on actual performance